In [1]:
import re
import unicodedata
import requests
from bs4 import BeautifulSoup
from fpdf import FPDF
from pathlib import Path

C:\Users\Mehdi\AppData\Roaming\Python\Python313\site-packages\fpdf\__init__.py:41: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


In [4]:
#data documentations
data_urls = [
    ("https://spring.io/guides/gs/relational-data-access", "Accessing Relational Data using JDBC with Spring"),
    ("https://spring.io/guides/gs/accessing-data-neo4j", "Accessing Data with Neo4j"),
    ("https://spring.io/guides/gs/managing-transactions", "Managing Transactions"),
    ("https://spring.io/guides/gs/accessing-data-jpa", "Accessing Data with JPA"),
    ("https://spring.io/guides/gs/accessing-data-mongodb", "Accessing Data with MongoDB"),
    ("https://spring.io/guides/gs/accessing-data-rest", "Accessing JPA Data with REST"),
    ("https://spring.io/guides/gs/accessing-mongodb-data-rest", "Accessing MongoDB Data with REST"),
    ("https://spring.io/guides/gs/accessing-data-mysql", "Accessing data with MySQL"),
    ("https://spring.io/guides/gs/spring-data-reactive-redis", "Accessing Data Reactively with Redis"),
    ("https://spring.io/guides/gs/accessing-data-r2dbc", "Accessing data with R2DBC"),
    ("https://spring.io/guides/gs/accessing-data-cassandra","Accessing Data with Cassandra")
]

urls = data_urls
data = []

In [2]:
#security documentations
security_urls = [
    ("https://spring.io/guides/gs/authenticating-ldap", "Authenticating a User with LDAP"),
    ("https://spring.io/guides/gs/securing-web", "Securing a Web Application"),
    ("https://spring.io/guides/gs/rest-service-cors", "Enabling Cross Origin Requests for a RESTful Web Service"),
    ("https://spring.io/guides/tutorials/spring-security-and-angular-js", "Spring Security and Angular"),
    ("https://spring.io/guides/tutorials/spring-boot-oauth2", "Spring Boot and OAuth2"),
    ("https://spring.io/guides/gs/accessing-vault", "Accessing Vault"),
    ("https://spring.io/guides/gs/vault-config", "Vault Configuration")
]

urls = security_urls
data = []

In [3]:
for url, title in urls:
    response = requests.get(url)
    if response.status_code != 200:
        print(f"FAILED ({response.status_code}): {url}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")

    for tag in soup.find_all(['nav', 'header', 'footer', 'script', 'style']):
        tag.decompose()

    sections = []
    current = {'heading': '', 'content': ''}

    for tag in soup.find_all(['h2', 'h3', 'p', 'pre', 'li']):
        if tag.name in ['h2', 'h3']:
            if current['content'].strip():
                sections.append(current)
            current = {'heading': tag.get_text(strip=True), 'content': ''}
        else:
            current['content'] += tag.get_text(separator=' ', strip=True) + '\n'

    if current['content'].strip():
        sections.append(current)

    data.append({'title': title, 'url': url, 'sections': sections})
    print(f"OK  {title}  ({len(sections)} sections)")

OK  Authenticating a User with LDAP  (13 sections)
OK  Securing a Web Application  (11 sections)
OK  Enabling Cross Origin Requests for a RESTful Web Service  (16 sections)
OK  Spring Security and Angular  (64 sections)
OK  Spring Boot and OAuth2  (27 sections)
OK  Accessing Vault  (16 sections)
OK  Vault Configuration  (14 sections)


In [4]:
def sanitize(text):
    normalized = unicodedata.normalize('NFKD', text)
    return normalized.encode('ascii', errors='ignore').decode('ascii')


def save_as_pdf(title, url, sections, output_path):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Title
    pdf.set_font("Helvetica", 'B', 16)
    pdf.multi_cell(0, 10, sanitize(title))
    pdf.ln(2)

    # Source URL
    pdf.set_font("Helvetica", 'I', 9)
    pdf.multi_cell(0, 6, f"Source: {url}")
    pdf.ln(3)

    # Divider line
    y = pdf.get_y()
    pdf.line(10, y, 200, y)
    pdf.ln(6)

    # Sections
    for section in sections:
        if section['heading']:
            pdf.set_font("Helvetica", 'B', 12)
            pdf.multi_cell(0, 8, sanitize(section['heading']))
            pdf.ln(1)
        if section['content'].strip():
            pdf.set_font("Helvetica", '', 10)
            content = re.sub(r'\n{3,}', '\n\n', section['content'].strip())
            pdf.multi_cell(0, 6, sanitize(content))
        pdf.ln(4)

    pdf.output(str(output_path))


output_dir = Path("../data/raw/security")
output_dir.mkdir(parents=True, exist_ok=True)

for entry in data:
    filename = entry['title'].replace(' ', '_') + '.pdf'
    path = output_dir / filename
    save_as_pdf(entry['title'], entry['url'], entry['sections'], path)
    print(f"Saved: {path}")

Saved: ..\data\raw\security\Authenticating_a_User_with_LDAP.pdf
Saved: ..\data\raw\security\Securing_a_Web_Application.pdf
Saved: ..\data\raw\security\Enabling_Cross_Origin_Requests_for_a_RESTful_Web_Service.pdf
Saved: ..\data\raw\security\Spring_Security_and_Angular.pdf
Saved: ..\data\raw\security\Spring_Boot_and_OAuth2.pdf
Saved: ..\data\raw\security\Accessing_Vault.pdf
Saved: ..\data\raw\security\Vault_Configuration.pdf
